In [1]:
using JuMP
using Gurobi
using Random
import XLSX
import JSON
using DataFrames
using CSV

gurobi_solver = JuMP.optimizer_with_attributes(Gurobi.Optimizer, "FeasibilityTol"=>1e-6)

# Design of Experiment
demand_status = "D2" # "D1" => "D_low" or "D2" => "D_med" or "D3" => "D_high"
supply_status = "S2" # "S1" => "S_low" or "S2" => "S_med" or "S3" => "S_high"
price_status = "P1" # "P1" => "P_no_discount"
overlap_decision = true
capacity_extension_decision = true

################################################### INDICES ####################################################
#=
Index Definitions:
A: Set of antigens
V: Set of vaccines
A_v: Subset of antigens in vaccine v
V_a: Subset of vaccines in antigen a
P: Set of producers
P_v: Subset of producers of vaccine v
T: Set of time periods
=#
#println("antigens") (single antigen only)
A = ["Measles","Mumps","Rubella","Diphtheria","Tetanus","Pertussis","Hepatitis_B","Hib","Polio","HPV","Rotavirus","PCV"]
#println("vaccines")
V = ["M","MR","MMR","TT","HepB","Hib","IPV","OPV","DT","Td","DTwP","DTwP-Hib","Penta","Hexa","HPV","Rotavirus","PCV"]
#println("vaccine,antigen dict")
A_v = Dict("M" => ["Measles"],"MR" => ["Measles","Rubella"],"MMR" => ["Measles","Mumps","Rubella"], "TT" => ["Tetanus"], "HepB" => ["Hepatitis_B"], "Hib" => ["Hib"], "IPV" => ["Polio"], 
            "OPV" => ["Polio"], "DT" => ["Diphtheria","Tetanus"], "Td" => ["Diphtheria","Tetanus"], "DTwP" => ["Diphtheria","Tetanus","Pertussis"],
            "DTwP-Hib" => ["Diphtheria","Tetanus","Pertussis","Hib"], "Penta" => ["Diphtheria","Tetanus","Pertussis","Hepatitis_B","Hib"], 
            "Hexa" => ["Diphtheria","Tetanus","Pertussis","Hepatitis_B","Hib","Polio"],"HPV" => ["HPV"], "Rotavirus" => ["Rotavirus"], "PCV" => ["PCV"])

V_a = Dict()
for a in A
    vector_a = []
    for v in keys(A_v)
        if a in A_v[v]
            push!(vector_a, v)
        end
    end
    V_a[a] = vector_a
end

P = ["AJ_Vaccines","BB_NCIPD","Beijing_Institute","Bharat_Biotech","Bilthoven","Biological_E","Centro_de","GSK","Haffkine_Bio",
        "LG_Chem","Merck_Sharp","Panacea_Biotec","PT_Bio","Sanofi_Pasteur","Serum_Institute","Xiamen_Innovax","Pfizer"]

P_v = Dict("M" => ["Serum_Institute", "PT_Bio"], "MR" => ["Serum_Institute", "Biological_E"], "MMR" => ["Serum_Institute","GSK","Merck_Sharp"],
    "TT"=> ["Serum_Institute","PT_Bio","BB_NCIPD"], "HepB" => ["Serum_Institute","LG_Chem"], "Hib" => ["Serum_Institute","Sanofi_Pasteur","Centro_de"], 
    "IPV" => ["LG_Chem","AJ_Vaccines","Bilthoven","Sanofi_Pasteur"], 
    "OPV" => ["Serum_Institute","PT_Bio","GSK","Sanofi_Pasteur","Panacea_Biotec","Beijing_Institute","Bharat_Biotech","Haffkine_Bio"],
    "DT" => ["Serum_Institute","PT_Bio","BB_NCIPD"], "Td" => ["Serum_Institute","PT_Bio","BB_NCIPD"], "DTwP" => ["Serum_Institute","Biological_E"], "DTwP-Hib" => ["Serum_Institute"],
    "Penta" => ["Serum_Institute","PT_Bio","Biological_E","LG_Chem","Panacea_Biotec"], "Hexa" => ["Sanofi_Pasteur"], 
    "HPV" => ["GSK","Merck_Sharp","Xiamen_Innovax"], "Rotavirus" => ["Serum_Institute","GSK","Bharat_Biotech"], "PCV" => ["Serum_Institute","GSK","Pfizer"])

V_p = Dict()
for p in P
    vector_p = []
    for v in keys(P_v)
        if p in P_v[v]
            push!(vector_p, v)
        end
    end
    V_p[p] = vector_p
end

P_a = Dict()
for a in A
    vector_a = []
    vaccines = V_a[a]
    for v in vaccines
        producers = P_v[v]
        for p in producers
            if p ∉ vector_a
                push!(vector_a, p)
            end
        end
    end
    P_a[a] = vector_a
end

A_p = Dict()
for p in P
    vector_p = []
    for a in keys(P_a)
        if p in P_a[a]
            push!(vector_p, a)
        end
    end
    A_p[p] = vector_p
end

tmin = 1
tmax = 10
T = [t for t in tmin:tmax]
T_initial = [t for t in tmin-1:tmax]

Δ = [1,2,3,4,5]

################################################### PARAMETERS ####################################################
#=
Parameter Definitions:
d: demand for antigen a at time t
s: production capacity of producer p at time t
k: max annual production batch size of vaccine v at time t
r: reservation price of vaccine v produced by p at time t
r_avg: average price of vaccine v in period t
l: annualized return on investment that producer p requires for vaccine v
gamma: maximum discount per dose achieved at highest allowed procurement quantity
g: set up cost if having a tender in period t (for GAVI)
f: production set-up cost of producer p for vaccine v in period t
h: annual holding cost for vaccine v as a proportion of price
pi: penalty for shortage of amount committed
beta: risk parameter for demand
=#

# source_1 = string("C:/Users/fthcn/Desktop/Vaccine_Tender_Scheduling_Problem/Demand_Scenarios_updated.xlsx")
# demand_file = XLSX.readxlsx(source_1)

# Get the absolute path of the current file's directory
current_directory = @__DIR__

# Define the file name
filename = "MVP_random_normal_forecast_data.xlsx"

# Construct the relative path using joinpath
relative_path = joinpath(current_directory, filename)

# Print the resulting path
println("Relative Path: ", relative_path)

demand_file = XLSX.readxlsx(relative_path)

d_real_raw = demand_file["Medium_Demand"]

total_demand_row = length(A)+1
total_demand_col = length(T)+1

d_real = Dict()
for row in 2:total_demand_row
    antigen = d_real_raw[row,1]
    for col in 2:total_demand_col
        year = d_real_raw[1,col]
        if demand_status == "D1"
            d_real[antigen,year] = 0.8*d_real_raw[row,col]
        elseif demand_status == "D2"
            d_real[antigen,year] = d_real_raw[row,col]
        elseif demand_status == "D3"
            d_real[antigen,year] = 1.2*d_real_raw[row,col]
        end
    end
end

Relative Path: C:\Users\nicho\OneDrive\Desktop\Vaccine_Tender\MVP_random_normal_forecast_data.xlsx


In [2]:
d_real

Dict{Any, Any} with 120 entries:
  ("Measles", 2)     => 2.48245e9
  ("Tetanus", 3)     => 3.05928e9
  ("Hib", 5)         => 1.90242e9
  ("Measles", 6)     => 2.63055e9
  ("Rotavirus", 7)   => 2.17777e9
  ("Mumps", 6)       => 1.92111e8
  ("Pertussis", 6)   => 2.05805e9
  ("Hib", 4)         => 1.57813e9
  ("Rubella", 1)     => 2.06802e9
  ("Measles", 8)     => 4.22225e9
  ("Hepatitis_B", 5) => 1.68678e9
  ("PCV", 5)         => 1.92057e9
  ("Diphtheria", 6)  => 6.87952e9
  ("Hib", 2)         => 1.4681e9
  ("Rotavirus", 8)   => 2.77676e9
  ("PCV", 7)         => 2.43242e9
  ("Diphtheria", 2)  => 3.4726e9
  ("Tetanus", 10)    => 4.07777e9
  ("Hepatitis_B", 4) => 2.12618e9
  ("Hepatitis_B", 7) => 1.87643e9
  ("Hib", 6)         => 1.819e9
  ("Hib", 9)         => 2.11075e9
  ("Hepatitis_B", 1) => 2.14966e9
  ("Polio", 3)       => 3.18397e9
  ("PCV", 10)        => 2.65754e9
  ⋮                  => ⋮

In [6]:
measles_dict = Dict(k => v for (k, v) in d_real if k[1] == "Measles")

Dict{Tuple{String, Int64}, Float64} with 10 entries:
  ("Measles", 2)  => 2.48245e9
  ("Measles", 7)  => 4.64312e9
  ("Measles", 9)  => 4.63317e9
  ("Measles", 6)  => 2.63055e9
  ("Measles", 5)  => 3.85623e9
  ("Measles", 8)  => 4.22225e9
  ("Measles", 10) => 5.49886e9
  ("Measles", 1)  => 2.19375e9
  ("Measles", 4)  => 3.0019e9
  ("Measles", 3)  => 2.22486e9

In [7]:
sorted_measles_dict = sort(collect(measles_dict), by = x -> x[1][2])

10-element Vector{Pair{Tuple{String, Int64}, Float64}}:
  ("Measles", 1) => 2.193753000435408e9
  ("Measles", 2) => 2.4824476056538076e9
  ("Measles", 3) => 2.224864354732567e9
  ("Measles", 4) => 3.001900115299745e9
  ("Measles", 5) => 3.856232647217439e9
  ("Measles", 6) => 2.630553742582485e9
  ("Measles", 7) => 4.643119414021532e9
  ("Measles", 8) => 4.2222545967234898e9
  ("Measles", 9) => 4.633165906472383e9
 ("Measles", 10) => 5.498863006795628e9

In [ ]:
    @objective(model, Min, sum(g[t]*F[a,(t,tau)]/(1+delta[t])^tmax for (t,tau) in F_time_set, a in A if (a,t,tau) ∉ starting_points_vect_F)
    + sum((r[v,p,t]*X[v,p,t]/(1+delta[t])^tmax) for v in V, p in P_v[v], t in T)
        + sum((pi*S[a,t]/(1+delta[t])^tmax) for a in A, t in T)
            + sum((h[v]*r_avg[v,t]*I[v,t]/(1+delta[t])^tmax) for v in V, t in T)
                + sum((Γ[p]*L[p,t]/(1+delta[t])^tmax) for p in P, t in T)
                                                            )

In [ ]:
    @objective(model, Min, sum(g[t]*F[a,(t,tau)]/(1+delta[t])^tmax for (t,tau) in F_time_set, a in A if (a,t,tau) ∉ starting_points_vect_F)
    + sum(p_ω[ω]*r[v,p,t]*X[v,p,t,ω]/(1+delta[t])^tmax for v in V, p in P_v[v], t in T, ω in Ω)
        + sum(p_ω[ω]*pi*S[a,t,ω]/(1+delta[t])^tmax for a in A, t in T, ω in Ω)
            + sum(p_ω[ω]*h[v]*r_avg[v,t]*I[v,t,ω]/(1+delta[t])^tmax for v in V, t in T, ω in Ω)
                + sum(Γ[p]*L[p,t]/(1+delta[t])^tmax for p in P, t in T)